In [2]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import torchvision.datasets as datasets

In [ ]:
mnist_train = datasets.MNIST(root="./data", train=True, download=True)
mnist_test = datasets.MNIST(root="./data", train=False, download=True)

X_train = mnist_train.data.float() / 255.0 # normalize between 0 and 1
y_train = mnist_train.targets.long()
X_test = mnist_test.data.float() / 255.0
y_test = mnist_test.targets.long()

In [4]:
# Convert into pytorch tensors
train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)

batch_size = 32
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
test_dl = DataLoader(test_ds, batch_size=test_ds.tensors[0].shape[0], shuffle=False)

In [ ]:
class MNIST_FNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 64) 
        self.fc2 = nn.Linear(64, 32) 
        self.fc3 = nn.Linear(32, 32)  
        self.fc4 = nn.Linear(32, 10)    

    def forward(self, x):
        x = torch.flatten(x, start_dim=1) # flatten the image, keep the batch dimension
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x

def train_one_epoch(model, train_dl, optimizer, loss_fn):
    model.train()
    batch_acc = []
    batch_loss = []
    for X_batch, y_batch in train_dl:
        output = model(X_batch)
        loss = loss_fn(output, y_batch)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        batch_acc_item = (output.argmax(dim=1) == y_batch).float().mean().item()
        batch_acc.append(batch_acc_item)
        batch_loss.append(loss.item())
        
    return np.mean(batch_acc), np.mean(batch_loss)

def train_model(model, train_dl, test_dl, optimizer, loss_fn, epochs):
    
    train_acc = np.zeros(epochs)
    test_acc = np.zeros(epochs)
    losses = np.zeros(epochs)
    
    for epoch in range(epochs):
        train_acc[epoch], losses[epoch] = train_one_epoch(model, train_dl, optimizer, loss_fn)
        
        model.eval()
        with torch.no_grad():
            for X_batch, y_batch in test_dl:
                output = model(X_batch)
                test_acc[epoch] = (output.argmax(dim=1) == y_batch).float().mean().item()
                
    return train_acc, test_acc, losses

In [17]:
# Experiment
learning_rates = np.logspace(np.log10(0.0001), np.log10(0.1), num=6)
epochs = 100

# IMPORTANT: the loss fun can be reused for multiple models, it doesnt store any state,
# but the optimizer needs to be redefined for each model since it stores the state of the parameters
loss_fn = nn.CrossEntropyLoss() # CrossEntropyLoss combines nn.LogSoftmax() and nn.NLLLoss()

optimizer_classes = {
    "SGD": torch.optim.SGD,
    "Adam": torch.optim.Adam,
    "RMSprop": torch.optim.RMSprop
}

test_accuracies = np.zeros((len(optimizer_classes), len(learning_rates)))

for i, (opt_name, opt_class) in enumerate(optimizer_classes.items()):
    for j, lr in enumerate(learning_rates):
        model = MNIST_FNN()
        optimizer = opt_class(model.parameters(), lr=lr)  
        train_acc, test_acc, losses = train_model(model, train_dl, test_dl, optimizer, loss_fn, epochs) 
        test_accuracies[i, j] = np.mean(test_acc[-10:])

KeyboardInterrupt: 

In [ ]:
plt.plot(learning_rates, test_accuracies[0], label="SGD")
plt.plot(learning_rates, test_accuracies[1], label="Adam")
plt.plot(learning_rates, test_accuracies[2], label="RMSprop")
plt.xscale("log")
plt.xlabel("Learning Rate")
plt.ylabel("Test Accuracy")
plt.title("Test Accuracy vs Learning Rate for Different Optimizers")
plt.legend()
plt.show()

# Adam and RMSprop, they are good at starting with low learning rates and then accelerates to compensate 
# for the large steps, but perform worse than SGD if the initial learning rate is high.